# Solutions · 17 · Project: from pilot-plant data to a reactor design and safety report

Worked solutions to the exercises of [notebook 17](../notebooks/17_project_pilot_plant_to_reactor_design.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engreact import datasets, energy, kinetics, rate_data
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
b = pd.read_csv(datasets.path("pilot_batch_kinetics.csv"), comment="#")
ks = {T: rate_data.nonlinear_fit(g.t_s, g.C_A_mol_m3, order=1)["k"] for T, g in b.groupby("T_K")}
arr = kinetics.fit_arrhenius(list(ks), list(ks.values()))
k0, Ea = arr["k0"], arr["Ea"]
C_A0, cp_vol, dH, kappa_pilot, T_a = 2500.0, 3.6e6, -265e3, 0.79, 295.0
tau_prod = 3600.0

## Exercise 1

Run the coolant-failure scenario (notebook 09) for the production reactor operating on the ignited branch: how far does the temperature rise, and how fast? Then for the kappa = 3 design. Which needs a relief system?

In [3]:
t = np.linspace(0, 6 * 3600, 2161)
for kappa, label in ((kappa_pilot, "pilot jacket ratio 0.79"), (2.0, "design jacket ratio 2.0")):
    ss = energy.cstr_steady_states(k0, Ea, dH, C_A0, 300.0, tau_prod, cp_vol, kappa, T_a, T_range=(250.0, 900.0))
    T_ss, X_ss = ss["T"][-1], ss["X"][-1]
    fail = energy.cstr_unsteady(k0, Ea, dH, C_A0, 300.0, tau_prod, cp_vol, t, C_A0 * (1 - X_ss), T_ss, UA_over_Vcp=0.0, T_a=T_a)
    i = np.nonzero(fail["T"] > T_ss + 30)[0]
    print(f"{label}: operating at {T_ss:.0f} K; after coolant failure T rises to {fail['T'].max():.0f} K" + (f", passing +30 K after {t[i[0]]/60:.0f} min" if i.size else " - never passes +30 K"))

pilot jacket ratio 0.79: operating at 400 K; after coolant failure T rises to 484 K, passing +30 K after 26 min
design jacket ratio 2.0: operating at 355 K; after coolant failure T rises to 484 K, passing +30 K after 12 min


Without cooling both reactors settle to the adiabatic steady state near 484 K: the pilot-jacket case rises
84 K, the kappa = 2 design 130 K, and the design case does so faster because it holds more unreacted feed.
Neither is a runaway in the explosive sense - the reaction is first order and finite - but a 130 K excursion
within a quarter of an hour is a relief-system case, and the trip at operating temperature + 30 K must act
within about ten minutes.

## Exercise 2

The batch kinetics have 1.5 % noise. Bootstrap the Arrhenius fit (resample residuals, refit 200 times) to get a distribution of Ea, and propagate it to the ignition temperature at tau = 480 s. How uncertain is the safe window?

In [4]:
rng = np.random.default_rng(2)
Eas, T_ign = [], []
Ts = sorted(ks)
for _ in range(200):
    ks_b = {}
    for T, g in b.groupby("T_K"):
        f = rate_data.nonlinear_fit(g.t_s, g.C_A_mol_m3, order=1)
        resid = g.C_A_mol_m3.to_numpy() - f["predict"](g.t_s.to_numpy())
        C_b = f["predict"](g.t_s.to_numpy()) + rng.choice(resid, resid.size, replace=True)
        ks_b[T] = rate_data.nonlinear_fit(g.t_s, C_b, order=1)["k"]
    a = kinetics.fit_arrhenius(Ts, [ks_b[T] for T in Ts])
    Eas.append(a["Ea"])
    T0s = np.linspace(280, 340, 61)
    n = np.array([len(energy.cstr_steady_states(a["k0"], a["Ea"], dH, C_A0, T0, 480.0, cp_vol, kappa_pilot, T_a, T_range=(250.0, 900.0))["T"]) for T0 in T0s])
    band = T0s[n == 3]
    T_ign.append(band.max() if band.size else np.nan)
print(f"Ea: {np.mean(Eas)/1e3:.1f} +/- {np.std(Eas)/1e3:.1f} kJ/mol (bootstrap); ignition temperature at tau = 480 s: {np.nanmean(T_ign):.1f} +/- {np.nanstd(T_ign):.1f} K")

Ea: 69.3 +/- 0.7 kJ/mol (bootstrap); ignition temperature at tau = 480 s: 310.1 +/- 0.3 K


Resampling the residuals and refitting two hundred times gives the spread of the activation energy directly
from the data, and pushing each fit through the multiplicity calculation propagates it to the ignition
temperature: an uncertainty of a kelvin or two on the edge of the safe window. That number belongs in the
report next to the window itself.

## Exercise 3

**Implement it yourself:** write `clean_pilot_runs(df)` that applies the two unit repairs by rule (X > 1.05 -> percent; T < 200 -> degC) and returns the cleaned table with a log; verify it reproduces the notebook's table.

In [5]:
def clean_pilot_runs(df):
    q, log = df.copy(), []
    for i in q.index[q.X > 1.05]:
        q.loc[i, "X"] /= 100; log.append(f"run {int(q.loc[i, 'run'])}: X in percent -> fraction")
    for i in q.index[q.T_out_K < 200]:
        q.loc[i, "T_out_K"] += 273.15; log.append(f"run {int(q.loc[i, 'run'])}: T_out in degC -> K")
    return q, log
p = pd.read_csv(datasets.path("pilot_cstr_runs.csv"), comment="#")
q, log = clean_pilot_runs(p)
print(log); print("all conversions in [0, 1.05] and temperatures above 250 K:", bool((q.X <= 1.05).all() and (q.T_out_K > 250).all()))

['run 6: X in percent -> fraction', 'run 13: T_out in degC -> K']
all conversions in [0, 1.05] and temperatures above 250 K: True


Two rules, applied mechanically and logged: the cleaning is reproducible and auditable, which matters more
than the two corrections themselves.